# 🎯 Stage 3: DPO Preference Optimization (Optional)
> Polish the model with Direct Preference Optimization to prefer better code, tool calls, and reasoning.

**Input Model:** Stage 2 merged model
**Method:** DPO (Direct Preference Optimization)
**Dataset:** Preference pairs (chosen vs rejected)
**GPU Required:** A100/H100
**Estimated Time:** 2-3 hours

### What is DPO?
Instead of just showing good examples (SFT), DPO shows **pairs**:
- Chosen: Good response (clean code, correct tool call, clear reasoning)
- Rejected: Bad response (buggy code, wrong function, sloppy reasoning)

The model learns to prefer good responses, refining output quality beyond SFT alone.

In [ ]:
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps trl peft accelerate bitsandbytes
!pip install -q datasets

In [ ]:
import torch
import json
import os
from unsloth import FastLanguageModel, PatchDPOTrainer
from datasets import Dataset
from trl import DPOTrainer, DPOConfig

PatchDPOTrainer()
print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
STAGE2_MODEL = "outputs/stage2_tool_calling/merged_model"
if not os.path.exists(STAGE2_MODEL):
    STAGE2_MODEL = "Qwen/Qwen2.5-Coder-7B-Instruct"
    print("Warning: Using base model. Run Stages 1-2 first for best results.")

MAX_SEQ_LENGTH = 2048
OUTPUT_DIR = "outputs/stage3_dpo"
print(f"Model: {STAGE2_MODEL}")

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=STAGE2_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model, r=8, lora_alpha=8,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                     "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0, bias="none",
    use_gradient_checkpointing="unsloth", random_state=42,
)
print("Model loaded with LoRA for DPO")

## Create DPO Preference Pairs

We generate preference pairs for three categories:
1. **Code Quality**: Clean vs buggy implementations
2. **Tool Calling**: Correct vs incorrect function invocations
3. **Reasoning**: Clear step-by-step vs jumbled explanations

In production, generate these from model outputs scored by tests or a reward model.
Below are starter examples. **Scale to 1,000-10,000+ pairs for best results.**

In [ ]:
# Example DPO preference pairs
dpo_pairs = [
    # Code Quality
    {
        "prompt": "Write a Python function to check if a string is a palindrome.",
        "chosen": "def is_palindrome(s: str) -> bool:\n    cleaned = ''.join(c.lower() for c in s if c.isalnum())\n    return cleaned == cleaned[::-1]",
        "rejected": "def is_palindrome(s):\n    return s == s[::-1]  # BUG: no case/punctuation handling"
    },
    # Tool Calling - correct use
    {
        "prompt": "What is the weather in San Francisco?",
        "chosen": "I will check the weather for you.\n<tool_call>\n{\"name\": \"get_weather\", \"arguments\": {\"city\": \"San Francisco\"}}\n</tool_call>",
        "rejected": "The weather in San Francisco is usually around 60-70F with fog. (I made this up without calling any tool.)"
    },
    # Tool Calling - should NOT call tool
    {
        "prompt": "Can you explain what a binary search tree is?",
        "chosen": "A BST is a data structure where each node has at most two children. Left subtree values are less than the node, right subtree values are greater. This enables O(log n) search, insert, and delete on average.",
        "rejected": "<tool_call>\n{\"name\": \"search_web\", \"arguments\": {\"query\": \"binary search tree\"}}\n</tool_call>\n(Wrong: should answer directly, not call a tool)"
    },
    # Reasoning Quality
    {
        "prompt": "Write a function to find the kth largest element in an array.",
        "chosen": "Step-by-step approach:\n1. Brute force: Sort and index -> O(n log n)\n2. Better: Min-heap of size k -> O(n log k)\n3. Optimal: Quickselect -> O(n) avg\n\nimport heapq\ndef kth_largest(nums, k):\n    return heapq.nlargest(k, nums)[-1]",
        "rejected": "def kth_largest(nums, k):\n    nums.sort()\n    return nums[-k]\n# No reasoning, no explanation"
    },
]

# Format for DPO trainer
formatted = []
for p in dpo_pairs:
    formatted.append({
        "prompt": f"<|im_start|>user\n{p['prompt']}<|im_end|>\n<|im_start|>assistant\n",
        "chosen": p["chosen"],
        "rejected": p["rejected"],
    })

dataset = Dataset.from_list(formatted)
print(f"Created {len(dataset)} DPO pairs")
print("For production quality, generate 1,000-10,000+ pairs.")

In [ ]:
dpo_trainer = DPOTrainer(
    model=model,
    ref_model=None,
    tokenizer=tokenizer,
    train_dataset=dataset,
    args=DPOConfig(
        output_dir=OUTPUT_DIR,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        num_train_epochs=3,
        learning_rate=5e-5,
        lr_scheduler_type="cosine",
        warmup_ratio=0.1,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=1,
        save_strategy="epoch",
        optim="adamw_8bit",
        seed=42,
        beta=0.1,
        max_length=MAX_SEQ_LENGTH,
        max_prompt_length=512,
        report_to="none",
    ),
)

print("Starting DPO training...")
trainer_stats = dpo_trainer.train()
print(f"DPO complete! Time: {trainer_stats.metrics['train_runtime']/60:.1f} min")

In [ ]:
# Save final model
model.save_pretrained(f"{OUTPUT_DIR}/lora_adapter")
tokenizer.save_pretrained(f"{OUTPUT_DIR}/lora_adapter")

merged = model.merge_and_unload()
merged.save_pretrained(f"{OUTPUT_DIR}/merged_model")
tokenizer.save_pretrained(f"{OUTPUT_DIR}/merged_model")

print(f"Final model: {OUTPUT_DIR}/merged_model")
print("Full pipeline complete! Run notebook 05 to benchmark, 06 to export GGUF.")